# Task 1-b — 절단 진단 및 판정기 비교

## 배경

직전 실험에서 문맥 범위(sent/prev1/prev2/full)에 따른 AUPRC 변동폭이 사전 기준 5%p에 미달했다.
그러나 이를 가설 기각으로 해석하기 전에, 판정기 자체가 판별력을 갖추고 있었는지 확인해야 한다.
직전 결과에서 RoBERTa-L-MNLI는 AUROC 0.578(full 조건 0.491)로 무작위와 구별되지 않았고,
DeBERTa-L-MNLI도 0.695에 그쳤다. 판별력이 없는 도구로 조건 간 차이를 비교하면 그 차이가
실제 효과인지 잡음인지 알 수 없다.

## 이 실험의 두 가지 질문

1. **절단 진단** — 근거 문서를 3000자로 자르고 토크나이저가 512토큰에서 다시 자르는 과정에서
   근거가 얼마나 소실되었는가. 소실이 크다면 판정기는 보지 못한 내용을 근거 없음으로 판정했을
   가능성이 있으며, 이 경우 어떤 모델을 써도 같은 결과가 나온다.
2. **판정기 비교** — 검증 단위는 문장(sent)으로 고정하고 판정기만 교체하여,
   미들웨어에 탑재할 만한 성능의 판정기가 존재하는지 확인한다.

## 사전 정의 기준

- **판별력 기준**: AUROC 0.80 이상
- **지연 기준**: 문장당 100ms 이하
- 두 기준을 함께 만족하는 기성 판정기가 없으면, RAGTruth로 직접 파인튜닝하는 방향으로 전환한다.

> 런타임 유형을 **GPU(T4)** 로 설정할 것. transformers 버전은 고정하지 않는다.

## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentencepiece scikit-learn pandas matplotlib
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git

import json, time, random, gc
import numpy as np, pandas as pd, torch, transformers
import pysbd
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device', DEVICE)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src, max_chars=None):
    """max_chars=None 이면 절단하지 않는다 (절단 효과를 측정하기 위함)."""
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        text = f"Question: {si.get('question','')}\n{si.get('passages','')}"
    elif tt == 'Summary':
        text = si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    else:
        parts = []
        for k, v in (si.items() if isinstance(si, dict) else []):
            parts.append(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}")
        text = '\n'.join(parts)
    return text if max_chars is None else text[:max_chars]

print('응답', len(responses), '| 소스', len(sources))

## 2. 절단 진단

직전 실험의 조건(3000자 절단 + 512토큰 절단)에서 근거가 얼마나 남았는지 측정한다.
모델은 로드하지 않고 토크나이저만 사용하므로 수 분 내에 끝난다.

In [ ]:
from transformers import AutoTokenizer
tok_diag = AutoTokenizer.from_pretrained('microsoft/deberta-large-mnli')

rows = []
for sid, src in sources.items():
    full = build_reference(src)                 # 절단 없음
    cut  = full[:3000]                          # 직전 실험의 1차 절단
    n_full = len(tok_diag(full,  add_special_tokens=False)['input_ids'])
    n_cut  = len(tok_diag(cut,   add_special_tokens=False)['input_ids'])
    rows.append({'source_id': sid, 'task_type': src['task_type'],
                 'chars_full': len(full), 'tok_full': n_full, 'tok_after_3000char': n_cut})
diag = pd.DataFrame(rows)

# 문장 가설이 평균 40토큰, 특수토큰 여유 10을 가정하면 근거에 쓸 수 있는 예산은 약 460토큰
BUDGET = 460
diag['tok_used']      = diag.tok_after_3000char.clip(upper=BUDGET)
diag['kept_ratio']    = diag.tok_used / diag.tok_full
diag['is_truncated']  = diag.tok_full > BUDGET

print('=== 과제 유형별 근거 문서 길이와 절단 정도 ===')
summary = diag.groupby('task_type').agg(
    n              = ('source_id','size'),
     median_tokens = ('tok_full','median'),
    p90_tokens    = ('tok_full', lambda s: s.quantile(0.9)),
    truncated_pct = ('is_truncated', lambda s: s.mean()*100),
    mean_kept_pct = ('kept_ratio',   lambda s: s.mean()*100),
).round(1)
print(summary)
print(f'\n전체 평균 잔존 비율: {diag.kept_ratio.mean()*100:.1f}%')
print(f'전체 절단 비율: {diag.is_truncated.mean()*100:.1f}%')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11,4))
for tt, g in diag.groupby('task_type'):
    ax[0].hist(g.tok_full.clip(upper=2500), bins=40, alpha=.55, label=tt)
ax[0].axvline(BUDGET, color='r', ls='--', label=f'budget {BUDGET}')
ax[0].set_xlabel('reference tokens'); ax[0].set_ylabel('count')
ax[0].set_title('Reference length distribution'); ax[0].legend()

summary.mean_kept_pct.plot(kind='bar', ax=ax[1], color='steelblue')
ax[1].set_ylabel('% of reference kept'); ax[1].set_title('Surviving reference after truncation')
ax[1].set_ylim(0,100); plt.tight_layout(); plt.show()

**해석 지침** — 특정 과제 유형의 잔존 비율이 낮다면, 직전 실험에서 그 유형의 판정 결과는
환각 탐지가 아니라 절단 여부를 반영했을 가능성이 크다. 아래 판정기 비교에서는 이를 피하기 위해
근거 문서를 창(window)으로 나누어 전체를 훑는 방식을 함께 평가한다.

## 3. 평가 표본 구성 (직전 실험과 동일 시드·동일 표본)

In [ ]:
N_PER_TASK = 120
dev_pool = [r for r in responses if r['split'] == 'train']
dev = []
for tt in ['QA', 'Summary', 'Data2txt']:
    cand = [r for r in dev_pool if r['task_type'] == tt]
    dev += random.sample(cand, min(N_PER_TASK, len(cand)))

seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def split_sentences(text):
    return [(s.start, s.end, text[s.start:s.end]) for s in seg.segment(text) if s.sent.strip()]

def label_sentence(a, b, labels, overlap_ratio=0.0):
    covered = sum(max(0, min(b, l['end']) - max(a, l['start'])) for l in labels)
    if covered == 0: return 0
    return 1 if covered / max(b-a, 1) >= overlap_ratio else 0

def dominant_label_type(a, b, labels):
    best, best_ov = None, 0
    for l in labels:
        ov = max(0, min(b, l['end']) - max(a, l['start']))
        if ov > best_ov: best, best_ov = l.get('label_type'), ov
    return best

recs = []
for r in dev:
    labels = r.get('labels') or []
    for i, (a, b, s) in enumerate(split_sentences(r['response'])):
        recs.append({'resp_id': r['id'], 'source_id': r['source_id'], 'task_type': r['task_type'],
                     'sent_idx': i, 'sentence': s,
                     'y': label_sentence(a, b, labels),
                     'label_type': dominant_label_type(a, b, labels)})
df = pd.DataFrame(recs)
df['reference_full'] = df.source_id.map(lambda sid: build_reference(sources[sid]))
df['reference_cut']  = df.reference_full.str[:3000]
print(f'문장 {len(df):,}개 | 양성 {df.y.sum():,}개 ({df.y.mean()*100:.1f}%)')
print(df.label_type.value_counts(dropna=False).to_dict())

## 4. 판정기 후보

| 판정기 | 성격 |
|---|---|
| DeBERTa-L-MNLI (절단) | 직전 실험 조건. 비교 기준선 |
| DeBERTa-L-MNLI (창 분할) | 같은 모델로 근거 전체를 훑음. 절단 가설 직접 검증 |
| HHEM-2.1-open | 사실 일관성 판정 전용 모델 |
| MiniCheck | 문서 대비 문장 단위 사실성 검증 전용. 긴 문서를 내부에서 분할 |

각 후보는 로드 실패 시 자동으로 건너뛴다.

In [ ]:
from transformers import AutoModelForSequenceClassification

class NLIJudge:
    """근거를 전제, 문장을 가설로 두고 (1 - 함의확률)을 환각 점수로 사용.
       window=True 이면 근거를 여러 창으로 나눠 각각 판정한 뒤 최대 함의확률을 취한다."""
    def __init__(self, ckpt='microsoft/deberta-large-mnli', name=None, window=False,
                 win_tokens=400, stride=200):
        self.name = name or ckpt.split('/')[-1]
        self.window, self.win_tokens, self.stride = window, win_tokens, stride
        self.tok = AutoTokenizer.from_pretrained(ckpt)
        self.m = AutoModelForSequenceClassification.from_pretrained(ckpt).to(DEVICE).eval()
        self.ent_idx = [i for i, l in self.m.config.id2label.items()
                        if l.lower().startswith('entail')][0]

    def _make_windows(self, text):
        ids = self.tok(text, add_special_tokens=False)['input_ids']
        if len(ids) <= self.win_tokens:
            return [text]
        out = []
        for i in range(0, len(ids), self.stride):
            chunk = ids[i:i+self.win_tokens]
            if not chunk: break
            out.append(self.tok.decode(chunk))
            if i + self.win_tokens >= len(ids): break
        return out

    @torch.no_grad()
    def _entail(self, prem, hyp):
        enc = self.tok(prem, hyp, truncation=True, max_length=512,
                       padding=True, return_tensors='pt').to(DEVICE)
        return self.m(**enc).logits.softmax(-1)[:, self.ent_idx].cpu().numpy()

    def score(self, pairs, batch=16):
        """pairs: [(reference, sentence)] → 환각 점수 배열"""
        if not self.window:
            out = []
            for i in range(0, len(pairs), batch):
                c = pairs[i:i+batch]
                out.extend(self._entail([p for p,_ in c], [h for _,h in c]).tolist())
            return 1.0 - np.array(out)
        # 창 분할: 문장마다 여러 창과 대조하고 최대 함의확률 사용
        flat_p, flat_h, owner = [], [], []
        for idx, (ref, sent) in enumerate(pairs):
            for w in self._make_windows(ref):
                flat_p.append(w); flat_h.append(sent); owner.append(idx)
        probs = []
        for i in range(0, len(flat_p), batch):
            probs.extend(self._entail(flat_p[i:i+batch], flat_h[i:i+batch]).tolist())
        best = np.zeros(len(pairs))
        for o, p in zip(owner, probs):
            best[o] = max(best[o], p)
        return 1.0 - best


class HHEMJudge:
    name = 'HHEM-2.1-open'
    def __init__(self):
        self.m = AutoModelForSequenceClassification.from_pretrained(
            'vectara/hallucination_evaluation_model', trust_remote_code=True).to(DEVICE).eval()
    @torch.no_grad()
    def score(self, pairs, batch=16):
        out = []
        for i in range(0, len(pairs), batch):
            s = self.m.predict(pairs[i:i+batch])
            s = s.detach().cpu().numpy() if torch.is_tensor(s) else np.asarray(s)
            out.extend(np.ravel(s).tolist())
        return 1.0 - np.array(out)


class MiniCheckJudge:
    name = 'MiniCheck-FlanT5-L'
    def __init__(self, model_name='flan-t5-large'):
        from minicheck.minicheck import MiniCheck
        self.s = MiniCheck(model_name=model_name, enable_prefix_caching=False)
    def score(self, pairs, batch=16):
        docs   = [p for p, _ in pairs]
        claims = [h for _, h in pairs]
        _, raw_prob, _, _ = self.s.score(docs=docs, claims=claims)
        return 1.0 - np.asarray(raw_prob)

In [ ]:
# MiniCheck는 별도 패키지 설치가 필요하다. 실패해도 나머지 실험은 진행된다.
!pip -q install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main" 2>/dev/null || echo 'MiniCheck 설치 실패 — 건너뜀'

## 5. 판정기 비교 실행 (검증 단위는 문장으로 고정)

In [ ]:
def evaluate(judge, ref_col, tag):
    pairs = list(zip(df[ref_col].tolist(), df.sentence.tolist()))
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    scores = judge.score(pairs)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    y = df.y.values
    return {
        'judge': tag,
        'AUROC': roc_auc_score(y, scores),
        'AUPRC': average_precision_score(y, scores),
        'ms_per_sentence': elapsed * 1000 / len(df),
    }, scores


CANDIDATES = [
    ('DeBERTa-MNLI (절단)',   lambda: NLIJudge(window=False), 'reference_cut'),
    ('DeBERTa-MNLI (창분할)', lambda: NLIJudge(window=True),  'reference_full'),
    ('HHEM-2.1-open',         HHEMJudge,                      'reference_cut'),
    ('MiniCheck-FlanT5-L',    MiniCheckJudge,                 'reference_full'),
]

results, all_scores = [], {}
for tag, factory, ref_col in CANDIDATES:
    try:
        judge = factory()
    except Exception as e:
        print(f'[skip] {tag}: {type(e).__name__}: {str(e)[:120]}')
        continue
    try:
        r, sc = evaluate(judge, ref_col, tag)
        results.append(r); all_scores[tag] = sc
        print(f"{tag:24} AUROC {r['AUROC']:.3f}  AUPRC {r['AUPRC']:.3f}  {r['ms_per_sentence']:7.1f} ms/문장")
    except Exception as e:
        print(f'[fail] {tag}: {type(e).__name__}: {str(e)[:120]}')
    finally:
        del judge; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

res = pd.DataFrame(results)
res

## 6. 사전 기준 판정

In [ ]:
AUROC_MIN, MS_MAX = 0.80, 100.0
RANDOM_AUPRC = df.y.mean()

print(f'무작위 기준선 AUPRC = {RANDOM_AUPRC:.3f}, AUROC = 0.500\n')
for _, r in res.iterrows():
    ok_acc  = r.AUROC >= AUROC_MIN
    ok_lat  = r.ms_per_sentence <= MS_MAX
    lift    = r.AUPRC / RANDOM_AUPRC
    verdict = '채택 가능' if (ok_acc and ok_lat) else ('정확도 미달' if not ok_acc else '지연 초과')
    print(f'{r.judge:24} AUROC {r.AUROC:.3f}{"✓" if ok_acc else "✗"}  '
          f'{r.ms_per_sentence:6.1f}ms{"✓" if ok_lat else "✗"}  '
          f'무작위 대비 AUPRC {lift:.2f}배 → {verdict}')

if not (res.AUROC >= AUROC_MIN).any():
    print('\n결론: 기준을 만족하는 기성 판정기가 없음 → RAGTruth 파인튜닝으로 전환')
else:
    best = res.loc[res.AUROC.idxmax()]
    print(f'\n결론: {best.judge} 채택 후보 → 이 판정기로 문맥 범위 실험을 재수행')

## 7. 오류 분석 — 어디서 실패하는가

과제 유형별 성능은 절단 진단 결과와 대조한다. 근거가 많이 잘린 유형에서 성능이 낮다면
절단이 원인임을 뒷받침한다. 환각 유형별 성능은 판정기의 기본 판별력을 보여준다.
명백한(Evident) 환각조차 잡지 못한다면 그 판정기는 이 과제에 부적합하다.

In [ ]:
if all_scores:
    best_tag = res.loc[res.AUROC.idxmax(), 'judge']
    a = df.copy(); a['score'] = all_scores[best_tag]
    print(f'분석 대상: {best_tag}\n')

    print('=== 과제 유형별 ===')
    for tt, g in a.groupby('task_type'):
        if g.y.nunique() < 2: continue
        print(f'  {tt:10} AUROC {roc_auc_score(g.y, g.score):.3f}  '
              f'AUPRC {average_precision_score(g.y, g.score):.3f}  (양성 {g.y.mean()*100:.1f}%)')

    print('\n=== 환각 유형별 (해당 유형 양성 vs 전체 음성) ===')
    neg = a[a.y == 0]
    for lt in [x for x in a.label_type.dropna().unique()]:
        pos = a[(a.y == 1) & (a.label_type == lt)]
        if len(pos) < 10: continue
        sub = pd.concat([pos, neg])
        print(f'  {lt:26} n={len(pos):4}  AUROC {roc_auc_score(sub.y, sub.score):.3f}')

    print('\n=== 문장 위치별 ===')
    a['pos'] = a.sent_idx.clip(upper=5)
    for p, g in a.groupby('pos'):
        if g.y.nunique() < 2: continue
        print(f'  {p}번째 문장  AUROC {roc_auc_score(g.y, g.score):.3f}  (n={len(g)})')

In [ ]:
diag.to_csv('truncation_diagnosis.csv', index=False)
res.to_csv('judge_comparison.csv', index=False)
print('저장: truncation_diagnosis.csv, judge_comparison.csv')
res